In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import glob, os
import time
import sys
sys.path.append('..')
from src.preprocessing import FeatureExtractor

### Чтение данных

In [2]:
folder_path = r'..\data\raw'
file_type = '/*csv'

files = glob.glob(folder_path + file_type)

latest_file = max(files, key=os.path.getctime)

time = latest_file[12:].split(sep="_")

access_time = pd.Timestamp(
    year=int(time[0][0:4]),
    month=int(time[0][4:6]),
    day=int(time[0][6:8]),
    hour=int(time[1][0:2]),
    minute=int(time[1][2:4]),
    tz='Europe/Moscow'
)

df = pd.read_csv(latest_file)
df =  df[df["price_byn"] > 30]

### Разбиение на тестовую и обучающую выборки

In [3]:
y = df["price_byn"]
X = df.drop(["price_byn"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

### Функция обучения и оценки пайплайна

In [4]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

# таблица с результатами
results = []


def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test):

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    cv = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error"
        },
        n_jobs=-1,
        return_train_score=False
    )

    results.append({
        "Model": name,

        "Test R2": r2,
        "Test RMSE": rmse,
        "Test MAE": mae,
        "Test MAPE": mape,

        "CV R2": cv["test_r2"].mean(),
        "CV RMSE": -cv["test_rmse"].mean(),
        "CV MAE": -cv["test_mae"].mean(),
    })


def show_results():

    df = pd.DataFrame(results)

    if df.empty:
        print("Нет результатов")
        return

    numeric_cols = df.select_dtypes(include=np.number).columns

    return (
        df.style
        .format("{:.4f}", subset=numeric_cols)

        # где больше — лучше
        .highlight_max(
            subset=["Test R2", "CV R2"],
            color="lightgreen"
        )

        # где меньше — лучше
        .highlight_min(
            subset=[
                "Test RMSE",
                "Test MAE",
                "Test MAPE",
                "CV RMSE",
                "CV MAE"
            ],
            color="lightgreen"
        )
    )

In [7]:
categorical_features = [
    "brand",
    "processor",
    "rom_type",
    "os", 
    "videocard",
    "videocard_brand",
    "region",
    "matrix_type",
    "display_resolution",
    "ram_type"
]

#=======================================================

cb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time)),
    ('regressor', CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=7,
        loss_function='MAE',
        early_stopping_rounds=50,
        random_seed=42,
        verbose=500,
        cat_features=categorical_features
    ))
])

evaluate_model(
    "CatBoost",
    cb_pipeline,
    X_train, y_train,
    X_test, y_test
)

#=======================================================

xgb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time, GBM_XGB=True)),
    ('regressor', XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        objective='reg:absoluteerror',
        random_state=42,
        verbosity=1,
        n_jobs=-1,
        enable_categorical=True
    ))
])

evaluate_model(
    "XGB",
    xgb_pipeline,
    X_train, y_train,
    X_test, y_test
)

#===========================================================

lgb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time, GBM_XGB=True)),
    ('regressor', LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        objective='mae', 
        random_state=42,
        verbosity=1,
        n_jobs=-1,
        enable_categorical=True
    ))
])

evaluate_model(
    "LGB",
    lgb_pipeline,
    X_train, y_train,
    X_test, y_test
)

0:	learn: 1233.6880293	total: 338ms	remaining: 5m 37s
500:	learn: 430.9912766	total: 58.2s	remaining: 58s
999:	learn: 396.6584974	total: 1m 55s	remaining: 0us
[LightGBM] [Warning] Unknown parameter: enable_categorical
[LightGBM] [Warning] Unknown parameter: enable_categorical
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000844 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 549
[LightGBM] [Info] Number of data points in the train set: 7370, number of used features: 17
[LightGBM] [Info] Start training from score 950.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

In [8]:
show_results()

,Model,Test R2,Test RMSE,Test MAE,Test MAPE,CV R2,CV RMSE,CV MAE
0,CatBoost,0.7547,974.7986,473.3637,0.5372,0.7267,1060.1630,484.8624
1,XGB,0.7696,944.8901,449.5513,0.5153,0.7291,1055.7057,472.1749
2,LGB,0.7692,945.6863,451.0279,0.5223,0.7302,1053.1835,468.0297


### Модель плохо сравляется с оценкой стоимости дорогих товаров.

In [10]:
y_pred = cb_pipeline.predict(X_test)
dif_column = abs(y_test-y_pred)
dif_df = pd.DataFrame({"test":y_test, "pred":y_pred, "dif":dif_column})
dif_df.sort_values("dif", ascending=False).head(30)

CatBoostError: There is no trained model to use predict(). Use fit() to train model. Then use this method.

In [14]:
df_middle = df[df["price_byn"] < 4000]

y = df_middle["price_byn"]
X = df_middle.drop(["price_byn"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [15]:
categorical_features = [
    "brand",
    "processor",
    "rom_type",
    "os", 
    "videocard",
    "videocard_brand",
    "region",
    "matrix_type",
    "display_resolution",
    "ram_type"
]

#=======================================================

cb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time)),
    ('regressor', CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=7,
        loss_function='MAE',
        early_stopping_rounds=50,
        random_seed=42,
        verbose=500,
        cat_features=categorical_features
    ))
])

evaluate_model(
    "CatBoost_middle",
    cb_pipeline,
    X_train, y_train,
    X_test, y_test
)

#=======================================================

xgb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time, GBM_XGB=True)),
    ('regressor', XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        objective='reg:absoluteerror',
        random_state=42,
        verbosity=1,
        n_jobs=-1,
        enable_categorical=True
    ))
])

evaluate_model(
    "XGB_middle",
    xgb_pipeline,
    X_train, y_train,
    X_test, y_test
)

#===========================================================

lgb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time, GBM_XGB=True)),
    ('regressor', LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        objective='mae', 
        random_state=42,
        verbosity=1,
        n_jobs=-1,
        enable_categorical=True
    ))
])

evaluate_model(
    "LGB_middle",
    lgb_pipeline,
    X_train, y_train,
    X_test, y_test
)

0:	learn: 774.2794412	total: 123ms	remaining: 2m 3s
500:	learn: 260.3856305	total: 1m 5s	remaining: 1m 5s
999:	learn: 236.2045288	total: 2m 8s	remaining: 0us
[LightGBM] [Warning] Unknown parameter: enable_categorical
[LightGBM] [Warning] Unknown parameter: enable_categorical
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000625 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 534
[LightGBM] [Info] Number of data points in the train set: 6621, number of used features: 17
[LightGBM] [Info] Start training from score 800.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warnin

In [16]:
show_results()

,Model,Test R2,Test RMSE,Test MAE,Test MAPE,CV R2,CV RMSE,CV MAE
0,CatBoost,0.7547,974.7986,473.3637,0.5372,0.7267,1060.1630,484.8624
1,XGB,0.7696,944.8901,449.5513,0.5153,0.7291,1055.7057,472.1749
2,LGB,0.7692,945.6863,451.0279,0.5223,0.7302,1053.1835,468.0297
3,CatBoost_middle,0.8075,595.4169,358.5015,0.4724,0.7995,599.7010,362.5016
4,XGB_middle,0.8157,582.5424,345.8502,0.4791,0.8122,580.5450,348.9003
5,LGB_middle,0.8211,573.9649,343.2711,0.4753,0.8106,582.9408,348.5150
6,CatBoost_middle,0.8003,448.6206,284.8250,0.4467,0.7799,476.1951,297.0857
7,XGB_middle,0.8195,426.4324,266.3051,0.4419,0.7903,464.4842,287.4157
8,LGB_middle,0.8183,427.9341,264.5419,0.4362,0.7905,464.5055,283.3181


Сравним метрики модели обученной на данных без ценового потолка, на данных с ценовым потолком с моделью обученной на обрезанных данных.